# The transverse-field Ising model

In this example we locate the quantum phase transition of the transverse-field Ising model, by scanning the field across it and comparing a finite chain with a calculation directly in the thermodynamic limit.
The Hamiltonian is given by

$$
H = -J (∑_{⟨i,j⟩} σᶻᵢ σᶻⱼ + g ∑ᵢ σˣᵢ)
$$

where $J$ sets the energy scale and the field $g$ tunes the competition between the $σᶻσᶻ$ interaction and the transverse $σˣ$ field.
The model has a quantum critical point at $g = 1$, which we will diagnose in three independent ways: the order parameter, the entanglement entropy and the correlation length.
To follow this example you need the following packages:

In [ ]:
using MPSKit, TensorKit, CairoMakie
# Model constructors for the MPSKit documentation and examples.
# Adapted from MPSKitModels.jl v0.4.7, src/models/{hamiltonians,transfermatrices}.jl:
# https://github.com/QuantumKitHub/MPSKitModels.jl/tree/v0.4.7/src/models
# Operators come from TensorKitTensors; chain assembly uses MPSKit directly.
#
# MIT License
# Copyright (c) 2021 Maarten Van Damme
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

module ExampleModels

using MPSKit, TensorKit
using TensorKitTensors: SpinOperators, BosonOperators, HubbardOperators

export chain_hamiltonian, transverse_field_ising, heisenberg_XXX
export hubbard_model, bose_hubbard_model, hard_hexagon

# L specifies an open finite chain; otherwise unitcell specifies an infinite chain.
function chain_hamiltonian(twosite, onesite = nothing; L = nothing, unitcell::Integer = 1)
    nsites = isnothing(L) ? unitcell : L
    nsites isa Integer && nsites > 0 || throw(ArgumentError("chain length must be a positive integer"))
    nbonds = isnothing(L) ? nsites : nsites - 1
    terms = Pair[(i, i + 1) => twosite for i in 1:nbonds]
    if !isnothing(onesite)
        append!(terms, [i => onesite for i in 1:nsites])
    end
    spaces = fill(space(twosite, 1), nsites)
    constructor = isnothing(L) ? InfiniteMPOHamiltonian : FiniteMPOHamiltonian
    return constructor(spaces, terms)
end

# H = -J sum(σz_i σz_{i+1} + g σx_i).
function transverse_field_ising(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, g = 1.0, kwargs...
    )
    ZZ = 4 * SpinOperators.S_z_S_z(T, symmetry; spin = 1 // 2)
    X = SpinOperators.σˣ(T, symmetry)
    return chain_hamiltonian(-J * ZZ, -J * g * X; kwargs...)
end
transverse_field_ising(symmetry::Type{<:Sector}; kwargs...) =
    transverse_field_ising(ComplexF64, symmetry; kwargs...)

# H = J sum(S_i ⋅ S_{i+1}); preserve the examples' spin-1 default.
function heisenberg_XXX(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, spin = 1, kwargs...
    )
    term = J * SpinOperators.S_exchange(T, symmetry; spin)
    return chain_hamiltonian(term; kwargs...)
end
heisenberg_XXX(symmetry::Type{<:Sector}; kwargs...) =
    heisenberg_XXX(ComplexF64, symmetry; kwargs...)

# H = -t sum(e⁺_i e⁻_{i+1} + h.c.) + U sum(n↑ n↓) - mu sum(n).
function hubbard_model(
        T::Type{<:Number} = ComplexF64, particle_symmetry::Type{<:Sector} = Trivial,
        spin_symmetry::Type{<:Sector} = Trivial;
        t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    # e_hopping includes the fermionic reordering sign and is Hermitian.
    hopping = HubbardOperators.e_hopping(T, particle_symmetry, spin_symmetry)
    interaction = HubbardOperators.ud_num(T, particle_symmetry, spin_symmetry)
    N = HubbardOperators.e_num(T, particle_symmetry, spin_symmetry)
    return chain_hamiltonian(-t * hopping, U * interaction - mu * N; kwargs...)
end

# H = -t sum(b⁺_i b⁻_{i+1} + h.c.) + U/2 sum(n(n-1)) - mu sum(n).
function bose_hubbard_model(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        cutoff::Integer = 5, t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    hopping = BosonOperators.b_plus_b_min(T, symmetry; cutoff) +
        BosonOperators.b_min_b_plus(T, symmetry; cutoff)
    N = BosonOperators.b_num(T, symmetry; cutoff)
    interaction = N * (N - id(domain(N)))
    return chain_hamiltonian(-t * hopping, U / 2 * interaction - mu * N; kwargs...)
end

function hard_hexagon(T::Type{<:Number} = ComplexF64)
    P = Vect[FibonacciAnyon](:τ => 1)
    O = ones(T, P ⊗ P ← P ⊗ P)
    block(O, FibonacciAnyon(:I)) .*= 0
    return InfiniteMPO([O])
end

end

using .ExampleModels
using TensorKitTensors.SpinOperators: σᶻ

For reproducibility of this page, we fix the seed of the random number generator:

In [ ]:
using Random
Random.seed!(1234);

We fix a finite chain length `L`, a bond dimension `D` and the set of field values to scan.
`D` is kept modest so the example runs quickly; increasing it sharpens the results in the thermodynamic limit.

In [ ]:
L = 16
D = 8
g_values = 0.1:0.1:2.0

## Order parameter

We compute the order parameter $|⟨σᶻ⟩|$ at every field value, both on a finite chain and in the thermodynamic limit.

For the finite chain we use an open chain of `L` sites and optimize with DMRG.
We average $⟨σᶻᵢ⟩$ over the sites and take the absolute value, since a symmetry-broken state can have either sign.

In [ ]:
ψ₀_finite = FiniteMPS(L, ℂ^2, ℂ^D)
M_finite = map(g_values) do g
    H = transverse_field_ising(; L, g)
    ψ, envs, info = find_groundstate(ψ₀_finite, H, DMRG(; verbosity = 0))
    return abs(sum(i -> expectation_value(ψ, i => σᶻ()), 1:L)) / L
end

In the thermodynamic limit, we drop the length `L` to build the Hamiltonian on the infinite chain, and optimize an `InfiniteMPS` with VUMPS.
We keep the optimized states, since we will reuse them for the entropy and the correlation length below.
On the ordered side, the infinite state settles into one of the two symmetry-broken ground states, so we again take the absolute value.

In [ ]:
ψ₀_infinite = InfiniteMPS(ℂ^2, ℂ^D)
states_infinite = map(g_values) do g
    H = transverse_field_ising(; g)
    ψ, envs, info = find_groundstate(ψ₀_infinite, H, VUMPS(; verbosity = 0))
    return ψ
end
M_infinite = [abs(expectation_value(ψ, 1 => σᶻ())) for ψ in states_infinite]
fig = Figure()
ax = Axis(fig[1, 1]; xlabel = "g", ylabel = "|⟨σᶻ⟩|")
scatter!(ax, g_values, M_finite; label = "L = $L")
scatter!(ax, g_values, M_infinite; label = "L = ∞")
vlines!(ax, [1.0]; color = :gray, linestyle = :dash)
axislegend(ax; position = :rt)
fig

The two calculations agree in the disordered phase, but not in the ordered one.
The infinite state stays ordered up to $g = 1$ and then collapses, locating the critical point.
On the finite chain, the magnetization vanishes well before $g = 1$.
The exact ground state of a finite chain is the symmetric superposition of the two ordered states, which only needs a small bond dimension, so DMRG readily finds it.
Spontaneous symmetry breaking only happens in the thermodynamic limit, which is why the order parameter is best measured there.

## Entanglement entropy

At a critical point correlations become long-ranged, and the ground state is more entangled than deep in either phase.
For an `InfiniteMPS`, `entropy` returns the entanglement entropy of every bond in the unit cell, which here contains a single site.

In [ ]:
S_infinite = [real(only(entropy(ψ))) for ψ in states_infinite]

fig, ax, _ = scatter(g_values, S_infinite; axis = (; xlabel = "g", ylabel = "S"))
vlines!(ax, [1.0]; color = :gray, linestyle = :dash)
fig

The entropy peaks near $g = 1$, the entanglement signature of the phase transition.

## Correlation length

Finally, the correlation length is extracted from the transfer matrix spectrum of the infinite state, and has no finite-chain analogue.
`correlation_length` returns one value per symmetry sector, and without symmetries there is only the trivial one.
It grows towards the critical point, so we plot it on a logarithmic scale.

In [ ]:
ξ_infinite = [only(values(correlation_length(ψ))) for ψ in states_infinite]

fig, ax, _ = scatter(g_values, ξ_infinite; axis = (; xlabel = "g", ylabel = "ξ", yscale = log10))
vlines!(ax, [1.0]; color = :gray, linestyle = :dash)
fig

The correlation length peaks near $g = 1$ as well.
At the critical point it should diverge, but an MPS with finite bond dimension `D` can only capture correlations up to a finite length, so the peak grows as `D` is increased.
The critical point itself is studied in more detail in the Ising CFT example, which matches its excitation spectrum to the predictions of conformal field theory.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*